In [ ]:
# pip install google-cloud-storage

In [ ]:
import json
import numpy as np
import io
from datetime import datetime, timedelta, timezone


### Upload some text to GCS

In [ ]:
from google.cloud import storage
client = storage.Client()
bucket = client.bucket("hyde-datalake")
blob = bucket.blob("permission_test/ok.txt")
blob.upload_from_string("Ok")

In [ ]:
# Note
# Bucket = storage container
# Blob   = one object (file) inside a bucket
# Blob   = Binary Large Object

In [1]:
import json
import numpy as np
import io
from datetime import datetime, timedelta, timezone

class GoogleCloudStorage:
    def __init__(self):
        self.client = storage.Client()

    ### ----------- Sharing base function ---------- ###
    def _get_or_create_bucket(self,bucket_name,location="asia-southeast1"):
        '''Checking Bucket exist on GCS'''
        try:
            bucket = self.client.get_bucket(bucket_name)
            print(f"Bucket exists  : {bucket_name}")
        except Exception:
            bucket = self.client.create_bucket(bucket_name, location=location)
            print(f"Bucket created : {bucket_name}")
        return bucket

    def blob_exists(self,bucket_name, blob_path) -> bool:
        '''check if object exists'''
        bucket = self._get_or_create_bucket(bucket_name)
        return bucket.blob(blob_path).exists()

    ### ---------- Read file function ----------- ###
    def read_json(self, bucket_name, blob_path):
        '''read json file'''
        bucket = self._get_or_create_bucket(bucket_name)
        blob   = bucket.blob(blob_path)
        return json.loads(blob.download_as_text())

    def read_text(self, bucket_name, blob_path):
        '''read text file'''
        bucket = self._get_or_create_bucket(bucket_name)
        blob   = bucket.blob(blob_path)
        return blob.download_as_text()

    def read_npy(self, bucket_name, blob_path):
        '''read .npy (embedding vector) file'''
        bucket = self._get_or_create_bucket(bucket_name)
        blob   = bucket.blob(blob_path)

        buffer = io.BytesIO()
        blob.download_to_file(buffer)
        buffer.seek(0)
        return np.load(buffer)

        
    ### ---------- Creation folder function ----------- ###
    def create_folder(self,bucket_name,folder_path):
        '''Creating folder and sub folder'''
        bucket = self._get_or_create_bucket(bucket_name)
        if not folder_path.endswith("/"):
            folder_path += "/"
        blob = bucket.blob(folder_path)
        blob.upload_from_string("")
        print(f"Folder created : gs://{bucket_name}/{folder_path}")

    ### ---------- Upload folder function ----------- ###
    def upload_json(self,bucket_name,blob_path,json_data):
        '''upload json file to bucket'''
        bucket = self._get_or_create_bucket(bucket_name)
        blob   = bucket.blob(blob_path)
        
        blob.upload_from_string(
            json.dumps(json_data,ensure_ascii = False),
            content_type = "application/json"
        )
        print(f"uploaded JSON -> gs://{bucket.name}/{blob_path}")

    def upload_text(self,bucket_name, blob_path, text_data):
        '''upload text file to bucket'''
        bucket = self._get_or_create_bucket(bucket_name)
        blob   = bucket.blob(blob_path)

        blob.upload_from_string(
            text_data,
            content_type = "text/plain"
        )
        print(f"Uploaded text -> gs://{bucket.name}/{blob_path}")

    def upload_npy(self,bucket_name, blob_path, array):
        '''upload embedding vector'''
        buffer = io.BytesIO()
        np.save(buffer, array)
        buffer.seek(0)
        
        bucket = self._get_or_create_bucket(bucket_name)
        blob = bucket.blob(blob_path)
        blob.upload_from_file(
            buffer,
            content_type = "application/octet-stream"
        )
        print(f"Uploaded NPY -> gs://{bucket.name}/{blob_path}")

    ### ---------- Read file function ----------- ###
    def read_json(self, bucket_name, blob_path):
        '''read json file'''
        bucket = self._get_or_create_bucket(bucket_name)
        blob   = bucket.blob(blob_path)
        return json.loads(blob.download_as_text())

    def read_text(self, bucket_name, blob_path):
        '''read text file'''
        bucket = self._get_or_create_bucket(bucket_name)
        blob   = bucket.blob(blob_path)
        return blob.download_as_text()

    def read_npy(self, bucket_name, blob_path):
        '''read .npy (embedding vector) file'''
        bucket = self._get_or_create_bucket(bucket_name)
        blob   = bucket.blob(blob_path)

        buffer = io.BytesIO()
        blob.download_to_file(buffer)
        buffer.seek(0)
        return np.load(buffer)

    ### ---------- Remove function ----------- ###
    def delete_blob(self, bucket_name, blob_path):
        bucket = self._get_or_create_bucket(bucket_name)
        blob   = bucket.blob(blob_path)
        if blob.exists():
            blob.delete()
        print(f"Deleted: gs://{bucket_name}/{blob_path}")

    def delete_folder(self, bucket_name, folder_path):
        '''Remove nest blob(file) in folder'''
        bucket = self._get_or_create_bucket(bucket_name)
        if not folder_path.endswith("/"):
            folder_path += "/"
        blobs = bucket.list_blobs(prefix=folder_path)
        count = 0
        for blob in blobs:
            blob.delete()
            count += 1
    
        print(f"Deleted {count} objects under gs://{bucket_name}/{folder_path}")

    def delete_by_ttl(self, bucket_name, prefix, ttl: timedelta):
        '''Remove folder with setting time
        timeformat support
        timedelta(
            days=...,
            seconds=...,
            microseconds=...,
            milliseconds=...,
            minutes=...,
            hours=...,
            weeks=...
        )
        '''
        bucket = self._get_or_create_bucket(bucket_name)
        now    = datetime.now(timezone.utc)
        blobs  = bucket.list_blobs(prefix=prefix)
        deleted = 0
        for blob in blobs:
            if blob.time_created and now - blob.time_created > ttl:
                blob.delete()
                deleted += 1
        print(f"TTL cleanup deleted {deleted} objects under {prefix}")
        
gcs = GoogleCloudStorage()

NameError: name 'storage' is not defined

In [ ]:
gcs.delete_by_ttl(
    bucket_name = "hyde-datalake",
    prefix="student001/",
    ttl=timedelta(minutes=5)     # 5 min
)

In [ ]:
gcs.delete_by_ttl(
    bucket_name = "hyde-datalake",
    prefix="student001/",
    ttl=timedelta(hours=24)     # 24 hours ~ 1 day
)

In [ ]:
gcs.delete_by_ttl(
    bucket_name = "hyde-datalake",
    prefix="student001/",
    ttl=timedelta(days=30)     # 30 days ~ 1 month
)

In [ ]:
gcs.delete_by_ttl(
    bucket_name = "hyde-datalake",
    prefix="student001/",
    ttl=timedelta(weeks=48)    # 48 weeks ~ 1 years
)

In [ ]:
datetime.now(timezone.utc)

#### Create Bucket

In [ ]:
gcs._get_or_create_bucket(
    bucket_name = "hyde-datalake"
)

In [ ]:
###

#### Create folder

In [ ]:
gcs.create_folder(
    bucket_name = "hyde-datalake",
    folder_path = "student001/embedding/"
)

In [ ]:
gcs.create_folder(
    bucket_name = "hyde-datalake",
    folder_path = "student001/metadata/"
)

In [ ]:
###

In [ ]:
metadata_data = {
    "project":"HeDE",
    "version":"v1",
    "status" :"ok"
}

In [ ]:
gcs.upload_json(
    bucket_name = "hyde-datalake",
    blob_path   = "student001/metadata/metadata_data.json",
    json_data   = metadata_data
)

In [ ]:
###

In [ ]:
gcs.upload_text(
    bucket_name = "hyde-datalake",
    blob_path   = "student001/metadata/hello.txt",
    text_data   = "Hello GCS"
)

In [ ]:
###

In [ ]:
gcs.upload_npy(
    bucket_name = "hyde-datalake",
    blob_path   = "student001/embedding/embedding01.npy",
    array       = np.array([1,2,3,4,5])
)
gcs.upload_npy(
    bucket_name = "hyde-datalake",
    blob_path   = "student001/embedding/embedding02.npy",
    array       = np.array([1,2,3,4,5])
)
gcs.upload_npy(
    bucket_name = "hyde-datalake",
    blob_path   = "student001/embedding/embedding03.npy",
    array       = np.array([1,2,3,4,5])
)
gcs.upload_npy(
    bucket_name = "hyde-datalake",
    blob_path   = "student001/embedding/embedding04.npy",
    array       = np.array([1,2,3,4,5])
)
gcs.upload_npy(
    bucket_name = "hyde-datalake",
    blob_path   = "student001/embedding/embedding05.npy",
    array       = np.array([1,2,3,4,5])
)

In [ ]:
gcs.blob_exists(
    bucket_name = "hyde-datalake",
    blob_path   = "student001/embedding/embedding01.npy",
)

In [ ]:
###

In [ ]:
gcs.read_json(
    bucket_name = "hyde-datalake",
    blob_path   = "student001/metadata/metadata_data.json"
)

In [ ]:
gcs.read_text(
    bucket_name = "hyde-datalake",
    blob_path   = "student001/metadata/hello.txt"
)

In [ ]:
gcs.read_npy(
    bucket_name = "hyde-datalake",
    blob_path   = "student001/embedding/embedding01.npy"
)

In [ ]:
gcs.delete_blob(
    bucket_name = "hyde-datalake",
    blob_path   = "student001/metadata/hello.txt"
)

In [ ]:
gcs.delete_folder(
    bucket_name = "hyde-datalake",
    folder_path   = "permission_test" 
)


<hr>

In [2]:
# client = storage.Client()
# bucket = client.bucket("hyde-datalake")
# blob = bucket.blob("permission_test/ok.txt")
# blob.upload_from_string("Ok")

In [41]:
import json
import numpy as np
import io
from datetime import datetime, timedelta, timezone
from google.cloud import storage

class GoogleCloudStorage:
    def __init__(self,bucket_name):
        self.client = storage.Client()
        try:
            self.bucket = self.client.get_bucket(bucket_name)
            print(f"Bucket exists  : {bucket_name}")
        except Exception:
            self.bucket = self.client.create_bucket(bucket_name, location=location)
            print(f"Bucket created : {bucket_name}")
            
    def blob_exists(self, blob_path) -> bool:
        '''check if object exists'''
        return self.bucket.blob(blob_path).exists()

    ### ---------- Upload folder function ----------- ###
    def upload_json(self,blob_path,json_data):
        '''upload json file to bucket'''
        blob   = self.bucket.blob(blob_path)
        
        blob.upload_from_string(
            json.dumps(json_data,ensure_ascii = False),
            content_type = "application/json"
        )
        print(f"uploaded JSON -> gs://{self.bucket.name}/{blob_path}")

    def upload_text(self, blob_path, text_data):
        '''upload text file to bucket'''
        blob   = self.bucket.blob(blob_path)

        blob.upload_from_string(
            text_data,
            content_type = "text/plain"
        )
        print(f"Uploaded text -> gs://{self.bucket.name}/{blob_path}")

    def upload_npy(self, blob_path, array):
        '''upload embedding vector'''
        buffer = io.BytesIO()
        np.save(buffer, array)
        buffer.seek(0)
        
        blob = self.bucket.blob(blob_path)
        blob.upload_from_file(
            buffer,
            content_type = "application/octet-stream"
        )
        print(f"Uploaded NPY -> gs://{self.bucket.name}/{blob_path}")
        
    ### ---------- Read file function ----------- ###
    def read_json(self, blob_path):
        '''read json file'''
        blob   = self.bucket.blob(blob_path)
        return json.loads(blob.download_as_text())

    def read_text(self, blob_path):
        '''read text file'''
        blob   = self.bucket.blob(blob_path)
        return blob.download_as_text()

    def read_npy(self, blob_path):
        '''read .npy (embedding vector) file'''
        blob   = self.bucket.blob(blob_path)

        buffer = io.BytesIO()
        blob.download_to_file(buffer)
        buffer.seek(0)
        return np.load(buffer)
        
    ### ---------- Creation folder function ----------- ###
    def create_folder(self,folder_path):
        '''Creating folder and sub folder'''
        if not folder_path.endswith("/"):
            folder_path += "/"
        blob = bucket.blob(folder_path)
        blob.upload_from_string("")
        print(f"Folder created : gs://{self.bucket_name}/{folder_path}")
        
    ### ---------- Remove function ----------- ###
    def delete_blob(self, blob_path):
        blob   = self.bucket.blob(blob_path)
        if blob.exists():
            blob.delete()
        print(f"Deleted: gs://{self.bucket_name}/{blob_path}")

    def delete_folder(self, folder_path):
        '''Remove nest blob(file) in folder'''
        if not folder_path.endswith("/"):
            folder_path += "/"
        blobs = self.bucket.list_blobs(prefix=folder_path)
        count = 0
        for blob in blobs:
            blob.delete()
            count += 1
    
        print(f"Deleted {count} objects under gs://{self.bucket_name}/{folder_path}")

    def delete_by_ttl(self, prefix, ttl: timedelta):
        '''Remove folder with setting time
        timeformat support
        timedelta(
            days=...,
            seconds=...,
            microseconds=...,
            milliseconds=...,
            minutes=...,
            hours=...,
            weeks=...
        )
        '''
        now    = datetime.now(timezone.utc)
        blobs  = bucket.list_blobs(prefix=prefix)
        deleted = 0
        for blob in blobs:
            if blob.time_created and now - blob.time_created > ttl:
                blob.delete()
                deleted += 1
        print(f"TTL cleanup deleted {deleted} objects under {prefix}")
        
cgs = GoogleCloudStorage(bucket_name = "hyde-datalake")

Bucket exists  : hyde-datalake


In [21]:
cgs.blob_exists(blob_path = "student001/embedding/")
cgs.blob_exists(blob_path = "student001/metadata/")

True

In [27]:
metadata_data = {
    "project":"HeDE",
    "version":"v1",
    "status" :"ok"
}
cgs.upload_json(
    blob_path   = "student001/metadata/metadata_data.json",
    json_data   = metadata_data
)

uploaded JSON -> gs://hyde-datalake/student001/metadata/metadata_data.json


In [32]:
cgs.upload_text(
    blob_path   = "student001/metadata/hello.txt",
    text_data   = "Hello GCS"
)

Uploaded text -> gs://hyde-datalake/student001/metadata/hello.txt


In [34]:
cgs.upload_npy(
    blob_path   = "student001/embedding/embedding01.npy",
    array       = np.array([1,2,3,4,5])
)
cgs.upload_npy(
    blob_path   = "student001/embedding/embedding02.npy",
    array       = np.array([1,2,3,4,5])
)
cgs.upload_npy(
    blob_path   = "student001/embedding/embedding03.npy",
    array       = np.array([1,2,3,4,5])
)
cgs.upload_npy(
    blob_path   = "student001/embedding/embedding04.npy",
    array       = np.array([1,2,3,4,5])
)
cgs.upload_npy(
    blob_path   = "student001/embedding/embedding05.npy",
    array       = np.array([1,2,3,4,5])
)

Uploaded NPY -> gs://hyde-datalake/student001/embedding/embedding01.npy
Uploaded NPY -> gs://hyde-datalake/student001/embedding/embedding02.npy
Uploaded NPY -> gs://hyde-datalake/student001/embedding/embedding03.npy
Uploaded NPY -> gs://hyde-datalake/student001/embedding/embedding04.npy
Uploaded NPY -> gs://hyde-datalake/student001/embedding/embedding05.npy


In [39]:
cgs.blob_exists(
    blob_path   = "student001/embedding/embedding01.npy",
)

True

In [42]:
cgs.read_json(
    blob_path   = "student001/metadata/metadata_data.json"
)

{'project': 'HeDE', 'version': 'v1', 'status': 'ok'}

In [43]:
cgs.read_text(
    blob_path   = "student001/metadata/hello.txt"
)

'Hello GCS'

In [44]:
cgs.read_npy(
    blob_path   = "student001/embedding/embedding01.npy"
)

array([1, 2, 3, 4, 5])

In [46]:
cgs = GoogleCloudStorage(bucket_name = "hyde-datalake-feeds")

Bucket exists  : hyde-datalake-feeds


<hr>

In [ ]:
    # def read_json(self, bucket_name, blob_path):
    #     '''read json file'''
    #     bucket = self._get_or_create_bucket(bucket_name)
    #     blob   = bucket.blob(blob_path)
    #     return json.loads(blob.download_as_text())

    # def read_text(self, bucket_name, blob_path):
    #     '''read text file'''
    #     bucket = self._get_or_create_bucket(bucket_name)
    #     blob   = bucket.blob(blob_path)
    #     return blob.download_as_text()

    # def read_npy(self, bucket_name, blob_path):
    #     '''read .npy (embedding vector) file'''
    #     bucket = self._get_or_create_bucket(bucket_name)
    #     blob   = bucket.blob(blob_path)

    #     buffer = io.BytesIO()
    #     blob.download_to_file(buffer)
    #     buffer.seek(0)
    #     return np.load(buffer)